In [ ]:
USE `theia_pitching_db`;

-- Get counts for the last month for each joint table
WITH last_month_trials AS (
    SELECT DISTINCT t.session_trial 
    FROM `trials` t
    JOIN `sessions` s ON t.session = s.session
    WHERE s.date >= DATE_SUB(CURRENT_DATE, INTERVAL 1 MONTH)
)
SELECT 
    'joint_angles' as table_name, 
    COUNT(DISTINCT ja.session_trial) as unique_trials,
    COUNT(*) as total_rows,
    (SELECT COUNT(*) FROM last_month_trials lmt 
     LEFT JOIN `joint_angles` ja2 ON lmt.session_trial = ja2.session_trial 
     WHERE ja2.session_trial IS NULL) as missing_trials
FROM `joint_angles` ja
JOIN last_month_trials lmt ON ja.session_trial = lmt.session_trial
UNION
SELECT 
    'joint_forces' as table_name,
    COUNT(DISTINCT jf.session_trial) as unique_trials,
    COUNT(*) as total_rows,
    (SELECT COUNT(*) FROM last_month_trials lmt 
     LEFT JOIN `joint_forces` jf2 ON lmt.session_trial = jf2.session_trial 
     WHERE jf2.session_trial IS NULL) as missing_trials
FROM `joint_forces` jf
JOIN last_month_trials lmt ON jf.session_trial = lmt.session_trial
UNION
SELECT 
    'joint_moments' as table_name,
    COUNT(DISTINCT jm.session_trial) as unique_trials,
    COUNT(*) as total_rows,
    (SELECT COUNT(*) FROM last_month_trials lmt 
     LEFT JOIN `joint_moments` jm2 ON lmt.session_trial = jm2.session_trial 
     WHERE jm2.session_trial IS NULL) as missing_trials
FROM `joint_moments` jm
JOIN last_month_trials lmt ON jm.session_trial = lmt.session_trial
UNION
SELECT 
    'joint_velos' as table_name,
    COUNT(DISTINCT jv.session_trial) as unique_trials,
    COUNT(*) as total_rows,
    (SELECT COUNT(*) FROM last_month_trials lmt 
     LEFT JOIN `joint_velos` jv2 ON lmt.session_trial = jv2.session_trial 
     WHERE jv2.session_trial IS NULL) as missing_trials
FROM `joint_velos` jv
JOIN last_month_trials lmt ON jv.session_trial = lmt.session_trial;

-- Check trials and sessions for the last month
SELECT 
    COUNT(*) as total_trials,
    COUNT(DISTINCT t.session_trial) as unique_session_trials,
    COUNT(DISTINCT t.session) as unique_sessions
FROM `trials` t
JOIN `sessions` s ON t.session = s.session
WHERE s.date >= DATE_SUB(CURRENT_DATE, INTERVAL 1 MONTH);

In [ ]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
import logging
from datetime import datetime, timedelta
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_database_connection():
    """Create database connection"""
    return mysql.connector.connect(
        host="localhost",
        user="your_username",
        password="your_password",
        database="theia_pitching_db"
    )

def check_data_completeness(df, table_name, key_columns):
    """Check for missing or incomplete data"""
    logger.info(f"\nChecking data completeness for {table_name}:")
    
    # Check for null values
    null_counts = df.isnull().sum()
    if null_counts.any():
        logger.warning(f"Null values found in {table_name}:")
        logger.warning(null_counts[null_counts > 0])
    
    # Check for key column completeness
    for col in key_columns:
        if col in df.columns:
            missing = df[col].isnull().sum()
            if missing > 0:
                logger.warning(f"Missing values in key column {col}: {missing} rows")
    
    # Additional checks for time-series data
    if 'time' in df.columns:
        time_gaps = df.groupby('session_trial')['time'].apply(
            lambda x: x.diff().describe()
        )
        logger.info(f"Time series statistics:\n{time_gaps.mean()}")

def validate_join(df_before, df_after, join_type, key_columns):
    """Enhanced validation that no data was lost in join"""
    logger.info(f"\nValidating {join_type} join:")
    
    # Basic count checks
    before_counts = {
        'rows': len(df_before),
        'session_trials': df_before['session_trial'].nunique(),
        'times': df_before['time'].nunique() if 'time' in df_before.columns else None
    }
    
    after_counts = {
        'rows': len(df_after),
        'session_trials': df_after['session_trial'].nunique(),
        'times': df_after['time'].nunique() if 'time' in df_after.columns else None
    }
    
    logger.info(f"Rows before: {before_counts['rows']}, after: {after_counts['rows']}")
    logger.info(f"Trials before: {before_counts['session_trials']}, after: {after_counts['session_trials']}")
    
    # Check for lost trials
    if 'session_trial' in df_before.columns:
        lost_trials = set(df_before['session_trial'].unique()) - set(df_after['session_trial'].unique())
        if lost_trials:
            logger.warning(f"Lost trials in {join_type} join: {lost_trials}")
    
    # Check key columns
    for col in key_columns:
        if col in df_before.columns and col in df_after.columns:
            before_unique = df_before[col].nunique()
            after_unique = df_after[col].nunique()
            if before_unique != after_unique:
                logger.warning(f"Unique values changed for {col}: {before_unique} -> {after_unique}")
    
    return before_counts['rows'] == after_counts['rows']

def integrate_joint_data():
    """Main function to integrate joint data tables with time filter"""
    conn = get_database_connection()
    
    # Get date for one month ago
    one_month_ago = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    
    # Load sessions first to get valid session IDs
    logger.info("Loading sessions from last month...")
    sessions = pd.read_sql(f"""
        SELECT 
            session,
            date,
            level,
            lab,
            height_meters,
            mass_kilograms
        FROM `sessions`
        WHERE date >= '{one_month_ago}'
    """, conn)
    
    # Get valid trials for these sessions
    trials = pd.read_sql(f"""
        SELECT 
            session_trial,
            session,
            trial,
            pitch_type,
            handedness,
            filename
        FROM `trials`
        WHERE session IN ({','.join(map(str, sessions['session']))})
    """, conn)
    
    valid_session_trials = tuple(trials['session_trial'].unique())
    
    # Load joint tables with session_trial filter
    logger.info("Loading joint tables...")
    joint_angles = pd.read_sql(f"""
        SELECT * FROM `joint_angles`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_forces = pd.read_sql(f"""
        SELECT * FROM `joint_forces`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_moments = pd.read_sql(f"""
        SELECT * FROM `joint_moments`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    joint_velos = pd.read_sql(f"""
        SELECT * FROM `joint_velos`
        WHERE session_trial IN {valid_session_trials}
    """, conn)
    
    # Check completeness of base tables
    key_columns = ['session_trial', 'time']
    check_data_completeness(joint_angles, 'joint_angles', key_columns)
    check_data_completeness(joint_forces, 'joint_forces', key_columns)
    check_data_completeness(joint_moments, 'joint_moments', key_columns)
    check_data_completeness(joint_velos, 'joint_velos', key_columns)
    
    # Perform joins with enhanced validation
    # 1. Join angles and forces
    angles_forces = pd.merge(
        joint_angles,
        joint_forces,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(joint_angles, angles_forces, "angles-forces", key_columns)
    
    # 2. Add moments
    angles_forces_moments = pd.merge(
        angles_forces,
        joint_moments,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(angles_forces, angles_forces_moments, "moments", key_columns)
    
    # 3. Add velocities
    all_joint_data = pd.merge(
        angles_forces_moments,
        joint_velos,
        on=['session_trial', 'time'],
        how='inner'
    )
    validate_join(angles_forces_moments, all_joint_data, "velocities", key_columns)
    
    # 4. Add trials metadata
    joint_with_trials = pd.merge(
        all_joint_data,
        trials,
        on='session_trial',
        how='left'
    )
    validate_join(all_joint_data, joint_with_trials, "trials", 
                 key_columns + ['session', 'trial', 'pitch_type'])
    
    # 5. Finally add sessions metadata
    final_dataset = pd.merge(
        joint_with_trials,
        sessions,
        on='session',
        how='left'
    )
    validate_join(joint_with_trials, final_dataset, "sessions", 
                 key_columns + ['session', 'date', 'level'])
    
    # Final validation and summary
    logger.info("\nFinal Dataset Summary:")
    logger.info(f"Total rows: {len(final_dataset)}")
    logger.info(f"Unique trials: {final_dataset['session_trial'].nunique()}")
    logger.info(f"Unique sessions: {final_dataset['session'].nunique()}")
    logger.info(f"Date range: {final_dataset['date'].min()} to {final_dataset['date'].max()}")
    
    # Check for any missing data in critical columns
    critical_columns = ['session_trial', 'time', 'session', 'date', 'pitch_type']
    missing_data = final_dataset[critical_columns].isnull().sum()
    if missing_data.any():
        logger.warning("Missing data in critical columns:")
        logger.warning(missing_data[missing_data > 0])
    
    return final_dataset

if __name__ == "__main__":
    integrated_data = integrate_joint_data()
    
    # Generate detailed summary statistics
    summary_stats = {
        'total_rows': len(integrated_data),
        'unique_trials': integrated_data['session_trial'].nunique(),
        'unique_sessions': integrated_data['session'].nunique(),
        'pitch_types': integrated_data['pitch_type'].value_counts().to_dict(),
        'sessions_per_day': integrated_data.groupby('date')['session'].nunique().describe().to_dict(),
        'trials_per_session': integrated_data.groupby('session')['session_trial'].nunique().describe().to_dict()
    }
    
    print("\nDetailed Summary Statistics:")
    for key, value in summary_stats.items():
        print(f"\n{key}:")
        print(value)
    
    # Optional: Save to CSV or new database table
    # integrated_data.to_csv("integrated_joint_data_last_month.csv", index=False)

#include the phases of the pitch

# Updated with addition metrics like pitch speed

verified:

    Pitch Phases (events):
    The CTE pulls columns such as PKH_time, FP_v5_time, MER_time, BR_time, and MAD_time from the events table. These fields exist and are used to compute phase durations accurately.

    Energy Metrics (energetics):
    Your CTE selects the original shoulder and elbow energy columns as well as the additional lower‐limb fields (lead/rear ankle, knee, and hip energy transfer and generation). All these columns are present in the energetics table according to your metadata.

    Force Data (force_plates):
    The force_data CTE selects lead and rear force components (x, y, z, and magnitude) from the force_plates table. These fields are correctly named and available.

    Joint Loads (joint_forces and joint_moments):
    The join in the joint_loads CTE uses session_trial and time to combine force data with moment data. The required columns (e.g. elbow_force_x, shoulder_upper_arm_force_x, elbow_moment_x, shoulder_thorax_moment_x, etc.) are present and correctly referenced.

    Joint Angles and Velocities (joint_angles and joint_velos):
    In the biomech_with_phase CTE, joint_angles and joint_velos are joined on session_trial and time. The columns for shoulder, elbow, torso, and pelvis angles—as well as the corresponding velocities—are exactly as specified in your metadata.

    Trunk–Pelvis Dissociation Calculation:
    The computed column using ABS(ja.torso_angle_z - ja.pelvis_angle_z) is valid because both torso_angle_z and pelvis_angle_z exist in the joint_angles table.

    Phase Assignment:
    The CASE statement uses the pitch phases from the events table (joined via the pitch_phases CTE). The join condition (ja.session_trial = pp.session_trial) ensures the correct association between the event times and the joint data.

    Composite Scores (poi table):
    In the final_metrics CTE, the LEFT JOIN on the poi table using session_trial is valid. The poi table contains columns such as pitch_speed_mph and max_shoulder_internal_rotational_velo, which you reference.

In [ ]:
/*
================================================================================
-- NOTES / GOALS FOR THIS QUERY:
--
-- GOALS:
-- 1. Create a comprehensive biomechanical profile for each pitch.
-- 2. Segment the pitch into phases (e.g., Wind-Up, Stride, Arm Cocking, etc.)
-- 3. Pull kinetic, kinematic, and energy data to assess potential injury risk.
-- 4. Compute additional metrics (e.g., cumulative load, variability, trunk–pelvis
--    dissociation) for deeper analysis.
-- 5. Join composite injury risk scores (e.g., pitch speed, shoulder rotation velocity)
--    for cross-validation.
--
-- TABLES AND WHY THEY ARE PULLED IN:
-- • events: Provides key event timestamps (PKH, FP_v5, MER, BR, MAD) for segmentation.
-- • energetics: Supplies energy transfer/generation metrics from the upper and lower
--   limbs (shoulder, elbow, ankle, knee, hip) for kinetic chain analysis.
-- • force_plates: Offers ground reaction force data (lead and rear forces) to assess
--   load absorption and force transmission.
-- • joint_forces and joint_moments: Deliver joint load and moment data (e.g. elbow,
--   shoulder forces/moments) to quantify tissue stress.
-- • joint_angles and joint_velos: Provide kinematic data (angles and velocities) for
--   the shoulder, elbow, torso, and pelvis to evaluate movement patterns.
-- • poi: Contains composite injury risk scores (e.g., pitch speed, max shoulder internal
--   rotational velocity) to correlate mechanical metrics with performance.
--
-- METRICS PULLED IN:
-- - Phase durations (computed from events)
-- - Energy metrics: shoulder, elbow and lower limb energy transfer/generation.
-- - Joint kinematics: angles and velocities for shoulder, elbow, torso, and pelvis.
-- - Joint loads and moments: forces and moments from upper extremity joints.
-- - Ground reaction forces: lead and rear force magnitudes.
-- - Derived metrics: trunk–pelvis dissociation, cumulative lead force, variability of
--   shoulder rotation, and average phase durations.
-- - Composite injury risk scores: pitch_speed_mph and max_shoulder_internal_rotational_velo.
--
-- All joins use session_trial and time (where applicable) to ensure data is accurately
-- combined without losing any important information.
================================================================================
*/

USE `theia_pitching_db`;

-- 1. Pitch Phases CTE remains unchanged.
WITH pitch_phases AS (
    SELECT 
        e.session_trial,
        e.PKH_time AS phase_pkh,
        e.FP_v5_time AS phase_fp,
        e.MER_time AS phase_mer,
        e.BR_time AS phase_br,
        e.MAD_time AS phase_mad,
        -- Phase durations for later use
        (e.FP_v5_time - e.PKH_time) AS duration_pkh_to_fp,
        (e.MER_time - e.FP_v5_time) AS duration_fp_to_mer,
        (e.BR_time - e.MER_time) AS duration_mer_to_br,
        (e.MAD_time - e.BR_time) AS duration_br_to_mad
    FROM `events` e
),

-- 2. Energy Metrics CTE updated with additional lower limb energy columns.
energy_metrics AS (
    SELECT 
        session_trial,
        time,
        shoulder_energy_transfer,
        shoulder_energy_generation,
        elbow_energy_transfer,
        elbow_energy_generation,
        -- Additional energy metrics for lower limbs:
        lead_ankle_energy_transfer,
        lead_ankle_energy_generation,
        rear_ankle_energy_transfer,
        rear_ankle_energy_generation,
        lead_knee_energy_transfer,
        lead_knee_energy_generation,
        rear_knee_energy_transfer,
        rear_knee_energy_generation,
        lead_hip_energy_transfer,
        lead_hip_energy_generation,
        rear_hip_energy_transfer,
        rear_hip_energy_generation
    FROM `energetics`
),

-- 3. (EMG data removed for now)

-- 4. Force Data CTE remains unchanged.
force_data AS (
    SELECT
        session_trial,
        time,
        lead_force_x,
        lead_force_y,
        lead_force_z,
        lead_force_mag,
        rear_force_x,
        rear_force_y,
        rear_force_z,
        rear_force_mag
    FROM `force_plates`
),

-- 5. Joint Loads CTE remains unchanged.
joint_loads AS (
    SELECT
        jf.session_trial,
        jf.time,
        jf.elbow_force_x,
        jf.elbow_force_y,
        jf.elbow_force_z,
        jf.shoulder_upper_arm_force_x,
        jf.shoulder_upper_arm_force_y,
        jf.shoulder_upper_arm_force_z,
        jm.elbow_moment_x,
        jm.elbow_moment_y,
        jm.elbow_moment_z,
        jm.shoulder_thorax_moment_x,
        jm.shoulder_thorax_moment_y,
        jm.shoulder_thorax_moment_z
    FROM `joint_forces` jf
    JOIN `joint_moments` jm 
      ON jf.session_trial = jm.session_trial 
     AND jf.time = jm.time
),

-- 6. Updated Biomechanics with Phase CTE:
--    • Added trunk_pelvis_dissociation calculation.
--    • (EMG data join removed.)
biomech_with_phase AS (
    SELECT 
        ja.session_trial,
        ja.time,
        -- Joint angles and velocities
        ja.shoulder_angle_x,
        ja.shoulder_angle_y,
        ja.shoulder_angle_z,
        ja.elbow_angle_x,
        ja.elbow_angle_y,
        ja.elbow_angle_z,
        jv.shoulder_velo_x,
        jv.shoulder_velo_y,
        jv.shoulder_velo_z,
        jv.elbow_velo_x,
        jv.elbow_velo_y,
        jv.elbow_velo_z,
        -- Core kinematics
        ja.torso_angle_x,
        ja.torso_angle_y,
        ja.torso_angle_z,
        ja.pelvis_angle_x,
        ja.pelvis_angle_y,
        ja.pelvis_angle_z,
        jv.torso_velo_x,
        jv.torso_velo_y,
        jv.torso_velo_z,
        -- Energy metrics including additional lower limb values
        em.shoulder_energy_transfer,
        em.shoulder_energy_generation,
        em.elbow_energy_transfer,
        em.elbow_energy_generation,
        em.lead_ankle_energy_transfer,
        em.lead_ankle_energy_generation,
        em.rear_ankle_energy_transfer,
        em.rear_ankle_energy_generation,
        em.lead_knee_energy_transfer,
        em.lead_knee_energy_generation,
        em.rear_knee_energy_transfer,
        em.rear_knee_energy_generation,
        em.lead_hip_energy_transfer,
        em.lead_hip_energy_generation,
        em.rear_hip_energy_transfer,
        em.rear_hip_energy_generation,
        -- Joint loads and moments
        jl.elbow_force_x,
        jl.elbow_force_y,
        jl.elbow_force_z,
        jl.shoulder_upper_arm_force_x,
        jl.shoulder_upper_arm_force_y,
        jl.shoulder_upper_arm_force_z,
        jl.elbow_moment_x,
        jl.elbow_moment_y,
        jl.elbow_moment_z,
        jl.shoulder_thorax_moment_x,
        jl.shoulder_thorax_moment_y,
        jl.shoulder_thorax_moment_z,
        -- Ground reaction forces
        fd.lead_force_mag,
        fd.rear_force_mag,
        -- New computed column for trunk–pelvis dissociation
        ABS(ja.torso_angle_z - ja.pelvis_angle_z) AS trunk_pelvis_dissociation,
        -- Determine pitch phase based on time and event timestamps
        CASE 
            WHEN ja.time <= pp.phase_pkh THEN 'Wind-Up'
            WHEN ja.time <= pp.phase_fp THEN 'Stride'
            WHEN ja.time <= pp.phase_mer THEN 'Arm Cocking'
            WHEN ja.time <= pp.phase_br THEN 'Arm Acceleration'
            WHEN ja.time <= pp.phase_mad THEN 'Arm Deceleration'
            ELSE 'Follow Through'
        END AS pitch_phase,
        -- Bring in phase durations for later use
        pp.duration_pkh_to_fp,
        pp.duration_fp_to_mer,
        pp.duration_mer_to_br,
        pp.duration_br_to_mad
    FROM `joint_angles` ja
    JOIN pitch_phases pp 
      ON ja.session_trial = pp.session_trial
    JOIN `joint_velos` jv 
      ON ja.session_trial = jv.session_trial 
     AND ja.time = jv.time
    LEFT JOIN energy_metrics em 
      ON ja.session_trial = em.session_trial 
     AND ja.time = em.time
    LEFT JOIN joint_loads jl 
      ON ja.session_trial = jl.session_trial 
     AND ja.time = jl.time
    LEFT JOIN force_data fd 
      ON ja.session_trial = fd.session_trial 
     AND ja.time = fd.time
),

-- 7. Final Selection with added cumulative and variability metrics
--    and a join to composite injury risk scores (using the poi table as an example).
final_metrics AS (
    SELECT 
        bp.session_trial,
        bp.pitch_phase,
        COUNT(*) AS frames_in_phase,
        AVG(shoulder_angle_x) AS avg_shoulder_flex_ext,
        AVG(shoulder_angle_y) AS avg_shoulder_abd_add,
        AVG(shoulder_angle_z) AS avg_shoulder_rotation,
        AVG(elbow_angle_x) AS avg_elbow_flex_ext,
        MAX(ABS(shoulder_velo_z)) AS max_shoulder_rotation_speed,
        MAX(ABS(elbow_velo_x)) AS max_elbow_extension_speed,
        AVG(torso_angle_x) AS avg_torso_flex,
        AVG(torso_angle_y) AS avg_torso_lateral_tilt,
        MAX(ABS(torso_velo_z)) AS max_torso_rotation_speed,
        AVG(shoulder_energy_transfer) AS avg_shoulder_energy_transfer,
        MAX(shoulder_energy_generation) AS max_shoulder_energy_generation,
        AVG(elbow_energy_transfer) AS avg_elbow_energy_transfer,
        MAX(elbow_energy_generation) AS max_elbow_energy_generation,
        MAX(ABS(elbow_moment_y)) AS max_elbow_varus_moment,
        MAX(ABS(shoulder_thorax_moment_x)) AS max_shoulder_distraction_force,
        MAX(lead_force_mag) AS max_lead_leg_force,
        MAX(rear_force_mag) AS max_push_off_force,
        -- New cumulative load metric (example: cumulative lead force)
        SUM(lead_force_mag) AS cumulative_lead_force,
        -- Variability metric (example: standard deviation of shoulder rotation)
        STDDEV(shoulder_angle_z) AS shoulder_rotation_variability,
        -- Aggregated trunk–pelvis dissociation
        AVG(trunk_pelvis_dissociation) AS avg_trunk_pelvis_dissociation,
        -- Include average phase durations for reference
        AVG(duration_pkh_to_fp) AS avg_duration_pkh_to_fp,
        AVG(duration_fp_to_mer) AS avg_duration_fp_to_mer,
        AVG(duration_mer_to_br) AS avg_duration_mer_to_br,
        AVG(duration_br_to_mad) AS avg_duration_br_to_mad,
        -- Example composite injury risk scores from the poi table
        p.pitch_speed_mph,
        p.max_shoulder_internal_rotational_velo
    FROM biomech_with_phase bp
    LEFT JOIN poi p 
      ON bp.session_trial = p.session_trial
    GROUP BY bp.session_trial, bp.pitch_phase
)

-- Final result output.
SELECT * FROM final_metrics
ORDER BY session_trial, 
    CASE pitch_phase
        WHEN 'Wind-Up' THEN 1
        WHEN 'Stride' THEN 2
        WHEN 'Arm Cocking' THEN 3
        WHEN 'Arm Acceleration' THEN 4
        WHEN 'Arm Deceleration' THEN 5
        WHEN 'Follow Through' THEN 6
    END;
